# Assignment 7d: Structural Health Monitoring

# E/22/194

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

## Model

$$\Theta = \theta \in (0,1], \qquad Y_k = \theta K_{\mathrm{nominal}} e^{\epsilon_k}, \qquad \epsilon_k \sim \mathcal{N}(0,\sigma^2)$$

Taking logarithms,

$$\ln Y_k = \ln(\theta K_{\mathrm{nominal}}) + \epsilon_k$$

Therefore,

$$\ln Y_k \mid \Theta=\theta \sim \mathcal{N}\left(\ln(\theta K_{\mathrm{nominal}}), \sigma^2\right)$$

and

$$Y_k \mid \Theta=\theta \sim \operatorname{LogNormal}\left(\ln(\theta K_{\mathrm{nominal}}), \sigma^2\right)$$


## 1. Prior Belief Boundaries

The initial prior is

$$\Theta \sim \operatorname{Beta}(8, 1.5)$$

Its density is

$$f_\Theta^{(0)}(\theta) = \frac{1}{B(8,1.5)}\,\theta^{7}(1-\theta)^{0.5}, \qquad 0 < \theta < 1$$

### Prior mean

For $\Theta \sim \operatorname{Beta}(\alpha,\beta)$,

$$\mathbb{E}[\Theta] = \frac{\alpha}{\alpha+\beta}$$

Hence,

$$\mathbb{E}[\Theta^{(0)}] = \frac{8}{8+1.5} = \frac{8}{9.5} = \frac{16}{19} \approx 0.8421$$

Thus, the expected initial remaining stiffness is **84.21%**.

### Prior mode

Since $\alpha>1$ and $\beta>1$,

$$\operatorname{Mode}(\Theta) = \frac{\alpha-1}{\alpha+\beta-2}$$

Therefore,

$$\operatorname{Mode}(\Theta) = \frac{8-1}{8+1.5-2} = \frac{7}{7.5} \approx 0.9333$$

Because $\alpha = 8 > \beta = 1.5$, most prior probability lies near $\theta = 1$.

Hence, $\operatorname{Beta}(8, 1.5)$ represents an initially healthy component while retaining uncertainty.


In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

alpha_0 = 8.0
beta_0 = 1.5

theta_grid = np.linspace(0.01, 1.0, 5000)

prior_density = beta.pdf(
    theta_grid,
    a=alpha_0,
    b=beta_0
)

prior_mean = alpha_0 / (alpha_0 + beta_0)

prior_mode = (
    (alpha_0 - 1)
    /
    (alpha_0 + beta_0 - 2)
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior_density,
        mode="lines",
        name="Beta(8, 1.5)"
    )
)

fig.add_vline(
    x=prior_mean,
    line_dash="dash",
    annotation_text=f"Mean = {prior_mean:.4f}"
)

fig.add_vline(
    x=prior_mode,
    line_dash="dot",
    annotation_text=f"Mode = {prior_mode:.4f}"
)

fig.update_layout(
    title="Initial Prior for Remaining Structural Stiffness",
    xaxis_title="Remaining stiffness efficiency, \u03b8",
    yaxis_title="Probability density",
    template="plotly_white",
    width=950,
    height=550
)

fig.show()


## 2. Structural Likelihood Formulation

From $y_k = \theta K_{\mathrm{nominal}} e^{\epsilon_k}$, solve for $\epsilon_k$:

$$\epsilon_k = \ln\left(\frac{y_k}{\theta K_{\mathrm{nominal}}}\right)$$

The transformation derivative is

$$\left|\frac{d\epsilon_k}{dy_k}\right| = \frac{1}{y_k}$$

Since $\epsilon_k \sim \mathcal{N}(0,\sigma^2)$, the single-measurement likelihood is

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left[-\frac{1}{2\sigma^2}\left(\ln \frac{y_k}{\theta K_{\mathrm{nominal}}}\right)^2\right], \qquad y_k>0$$

### Joint likelihood

Assuming conditional independence,

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta)$$

Therefore,

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left[-\frac{1}{2\sigma^2}\left(\ln \frac{y_i}{\theta K_{\mathrm{nominal}}}\right)^2\right]$$

Equivalent form:

$$L(\mathbf{y}^{(k)} \mid \theta) = \frac{\exp\left[-\dfrac{1}{2\sigma^2}\displaystyle\sum_{i=1}^{k}\left(\ln \frac{y_i}{\theta K_{\mathrm{nominal}}}\right)^2\right]}{(\sigma\sqrt{2\pi})^k \displaystyle\prod_{i=1}^{k} y_i}$$

### Log-likelihood

$$\log L(\mathbf{y}^{(k)} \mid \theta) = -k\log(\sigma\sqrt{2\pi}) - \sum_{i=1}^{k}\log y_i - \frac{1}{2\sigma^2}\sum_{i=1}^{k}\left(\ln \frac{y_i}{\theta K_{\mathrm{nominal}}}\right)^2$$


## 3. Non-Conjugate Grid Update

At step $k-1$,

$$f_{k-1}(\theta) = f_{\Theta \mid \mathbf{Y}^{(k-1)}}\left(\theta \mid \mathbf{y}^{(k-1)}\right)$$

After observing $y_k$,

$$f_k(\theta) \propto L(y_k \mid \theta)\, f_{k-1}(\theta)$$

The normalized posterior is

$$f_k(\theta) = \frac{L(y_k \mid \theta)\, f_{k-1}(\theta)}{\displaystyle\int_0^1 L(y_k \mid s)\, f_{k-1}(s)\, ds}$$

For the initial update,

$$f_1(\theta) \propto \theta^7(1-\theta)^{0.5} \exp\left[-\frac{1}{2\sigma^2}\left(\ln \frac{y_1}{\theta K_{\mathrm{nominal}}}\right)^2\right]$$

A Beta density has kernel $\theta^{\alpha-1}(1-\theta)^{\beta-1}$. The log-normal likelihood introduces $\exp\left[-\frac{(\ln\theta)^2+\cdots}{2\sigma^2}\right]$.

Therefore,

$$\operatorname{Beta\ prior} \times \operatorname{LogNormal\ likelihood} \neq \operatorname{Beta\ posterior}$$

Hence, the posterior is non-conjugate and its normalizing integral has no standard closed form. Numerical integration is required.


## 4. Running Point Estimates

Let

$$f_k(\theta) = f_{\Theta \mid \mathbf{Y}^{(k)}}\left(\theta \mid \mathbf{y}^{(k)}\right)$$

### Posterior mean

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid \mathbf{y}^{(k)}] = \int_0^1 \theta\, f_k(\theta)\, d\theta$$

### MAP estimate

The MAP is not an integral. It is the posterior mode:

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname*{arg\,max}_{0<\theta\leq 1} f_k(\theta)$$

### Posterior variance

$$V_k = \int_0^1 \left(\theta - \widehat{\theta}_{\mathrm{Bayes}}^{(k)}\right)^2 f_k(\theta)\, d\theta$$

Posterior standard deviation:

$$s_k = \sqrt{V_k}$$


## 5. Algorithmic Grid Approximation

Choose $\theta_{\min}=0.01$, $\theta_{\max}=1$, and construct

$$\theta_j = \theta_{\min} + (j-1)\Delta\theta, \qquad j=1,\ldots,M$$

where

$$\Delta\theta = \frac{\theta_{\max}-\theta_{\min}}{M-1}$$

Using $\theta_{\min}>0$ prevents $\ln(0)$.

### Step 0: Evaluate the prior

$$q_j^{(0)} = \frac{1}{B(8,1.5)}\,\theta_j^7(1-\theta_j)^{0.5}$$

Normalize numerically:

$$Z_0 \approx \operatorname{trapz}\left(q^{(0)},\theta\right)$$

$$f_j^{(0)} = \frac{q_j^{(0)}}{Z_0}$$

Thus, $\operatorname{trapz}\left(f^{(0)},\theta\right) = 1$.

### Step $k$: Evaluate the new likelihood

For every grid point,

$$\ell_{kj} = L(y_k \mid \theta_j)$$

$$\ell_{kj} = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left[-\frac{1}{2\sigma^2}\left(\ln \frac{y_k}{\theta_j K_{\mathrm{nominal}}}\right)^2\right]$$

### Unnormalized update

$$q_j^{(k)} = \ell_{kj}\, f_j^{(k-1)}$$

### Trapezoidal normalization

$$Z_k \approx \operatorname{trapz}\left(q^{(k)},\theta\right)$$

$$f_j^{(k)} = \frac{q_j^{(k)}}{Z_k}$$

Then, $\operatorname{trapz}\left(f^{(k)},\theta\right) = 1$.

### Grid estimators

Posterior mean:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} \approx \operatorname{trapz}\left(\theta\, f^{(k)},\theta\right)$$

MAP:

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \theta_{\arg\max_j f_j^{(k)}}$$

Variance:

$$V_k \approx \operatorname{trapz}\left[\left(\theta - \widehat{\theta}_{\mathrm{Bayes}}^{(k)}\right)^2 f^{(k)},\theta\right]$$

### Numerically stable log update

$$\log q_j^{(k)} = \log f_j^{(k-1)} + \log L(y_k \mid \theta_j)$$

Let $m_k = \max_j \log q_j^{(k)}$. Then,

$$\widetilde{q}_j^{(k)} = \exp\left[\log q_j^{(k)} - m_k\right]$$

Finally,

$$f_j^{(k)} = \frac{\widetilde{q}_j^{(k)}}{\operatorname{trapz}(\widetilde{q}^{(k)},\theta)}$$

Subtracting $m_k$ prevents numerical underflow without changing the posterior shape.


## 6. Simulation and Sequential Tracking

Given

$$\theta_{\mathrm{true}} = 0.68, \qquad K_{\mathrm{nominal}} = 50.0\ \mathrm{kN/mm}, \qquad \sigma = 0.15, \qquad n = 15$$

Simulate $\epsilon_k \sim \mathcal{N}(0, 0.15^2)$,

$$y_k = 0.68(50)\, e^{\epsilon_k}$$


In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from scipy.stats import beta


# =========================================================
# 1. Settings
# =========================================================
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_measurements = 15

alpha_0 = 8.0
beta_0 = 1.5

theta_min = 0.01
theta_max = 1.00
grid_size = 5000

theta_grid = np.linspace(
    theta_min,
    theta_max,
    grid_size
)

milestone_steps = [0, 1, 2, 5, 10, 15]


# =========================================================
# 2. Initial Beta prior
# =========================================================
posterior_density = beta.pdf(
    theta_grid,
    a=alpha_0,
    b=beta_0
)

# Normalize on the numerical grid
posterior_density /= np.trapezoid(
    posterior_density,
    theta_grid
)


# =========================================================
# 3. Simulate the sensor stream
# =========================================================
epsilon_values = rng.normal(
    loc=0.0,
    scale=sigma,
    size=n_measurements
)

sensor_readings = (
    theta_true
    * K_nominal
    * np.exp(epsilon_values)
)


# =========================================================
# 4. Step-zero estimators
# =========================================================
initial_mean = np.trapezoid(
    theta_grid * posterior_density,
    theta_grid
)

initial_map = theta_grid[
    np.argmax(posterior_density)
]

initial_variance = np.trapezoid(
    (
        theta_grid - initial_mean
    )**2
    * posterior_density,
    theta_grid
)

initial_sd = np.sqrt(initial_variance)

steps = [0]
posterior_means = [initial_mean]
map_estimates = [initial_map]
posterior_variances = [initial_variance]
posterior_sds = [initial_sd]

milestone_densities = {
    0: posterior_density.copy()
}

records = []


# =========================================================
# 5. Sequential Bayesian updates
# =========================================================
for k, y_k in enumerate(
    sensor_readings,
    start=1
):

    # -----------------------------------------------------
    # Log-likelihood over the theta grid
    # -----------------------------------------------------
    log_likelihood = (
        -np.log(
            y_k
            * sigma
            * np.sqrt(2 * np.pi)
        )
        -
        (
            np.log(
                y_k
                /
                (
                    theta_grid
                    * K_nominal
                )
            )**2
        )
        /
        (2 * sigma**2)
    )

    # -----------------------------------------------------
    # Log posterior update
    # -----------------------------------------------------
    log_previous_posterior = np.log(
        posterior_density + 1e-300
    )

    log_unnormalized = (
        log_previous_posterior
        +
        log_likelihood
    )

    # Numerical stabilization
    log_unnormalized -= np.max(
        log_unnormalized
    )

    unnormalized_density = np.exp(
        log_unnormalized
    )

    # -----------------------------------------------------
    # Trapezoidal normalization
    # -----------------------------------------------------
    normalization_constant = np.trapezoid(
        unnormalized_density,
        theta_grid
    )

    posterior_density = (
        unnormalized_density
        /
        normalization_constant
    )

    # -----------------------------------------------------
    # Posterior mean
    # -----------------------------------------------------
    posterior_mean = np.trapezoid(
        theta_grid
        * posterior_density,
        theta_grid
    )

    # -----------------------------------------------------
    # MAP estimate
    # -----------------------------------------------------
    posterior_map = theta_grid[
        np.argmax(posterior_density)
    ]

    # -----------------------------------------------------
    # Posterior variance and SD
    # -----------------------------------------------------
    posterior_variance = np.trapezoid(
        (
            theta_grid
            - posterior_mean
        )**2
        * posterior_density,
        theta_grid
    )

    posterior_sd = np.sqrt(
        posterior_variance
    )

    # -----------------------------------------------------
    # Store results
    # -----------------------------------------------------
    steps.append(k)

    posterior_means.append(
        posterior_mean
    )

    map_estimates.append(
        posterior_map
    )

    posterior_variances.append(
        posterior_variance
    )

    posterior_sds.append(
        posterior_sd
    )

    if k in milestone_steps:
        milestone_densities[k] = (
            posterior_density.copy()
        )

    records.append({
        "Step": k,
        "Noise epsilon": epsilon_values[k - 1],
        "Sensor reading": y_k,
        "Posterior mean": posterior_mean,
        "MAP": posterior_map,
        "Posterior variance": posterior_variance,
        "Posterior SD": posterior_sd,
        "Mean error": abs(
            posterior_mean - theta_true
        ),
        "MAP error": abs(
            posterior_map - theta_true
        )
    })


# =========================================================
# 6. Results table
# =========================================================
results = pd.DataFrame(records)

display(
    results.round(5)
)

print(
    "Initial prior mean:",
    round(initial_mean, 4)
)

print(
    "Initial prior MAP:",
    round(initial_map, 4)
)

print(
    "Final posterior mean:",
    round(posterior_means[-1], 4)
)

print(
    "Final MAP estimate:",
    round(map_estimates[-1], 4)
)

print(
    "Final posterior variance:",
    round(posterior_variances[-1], 6)
)

print(
    "Final posterior SD:",
    round(posterior_sds[-1], 4)
)


# =========================================================
# 7. Confidence criterion
# =========================================================
# Confidence definition:
# |mean - true| <= 0.02
# |MAP  - true| <= 0.02
# posterior SD <= 0.05

confidence_step = None

for k in range(1, n_measurements + 1):

    mean_condition = (
        abs(
            posterior_means[k]
            - theta_true
        )
        <= 0.02
    )

    map_condition = (
        abs(
            map_estimates[k]
            - theta_true
        )
        <= 0.02
    )

    sd_condition = (
        posterior_sds[k]
        <= 0.05
    )

    if (
        mean_condition
        and map_condition
        and sd_condition
    ):
        confidence_step = k
        break

print(
    "First confidence step:",
    confidence_step
)


# =========================================================
# 8. Approximate final posterior CDF
# =========================================================
cdf = np.concatenate(
    (
        [0.0],
        np.cumsum(
            (
                posterior_density[:-1]
                +
                posterior_density[1:]
            )
            * 0.5
            * np.diff(theta_grid)
        )
    )
)

cdf /= cdf[-1]

lower_95 = np.interp(
    0.025,
    cdf,
    theta_grid
)

upper_95 = np.interp(
    0.975,
    cdf,
    theta_grid
)

print(
    "Final 95% credible interval:",
    (
        round(lower_95, 4),
        round(upper_95, 4)
    )
)


# =========================================================
# 9. Safety-threshold probability
# =========================================================
theta_critical = 0.75

probability_below_threshold = np.interp(
    theta_critical,
    theta_grid,
    cdf
)

print(
    "P(theta < 0.75 | data):",
    round(
        probability_below_threshold,
        4
    )
)


# =========================================================
# 10. Plot 1: posterior density milestones
# =========================================================
fig_density = go.Figure()

for step in milestone_steps:

    fig_density.add_trace(
        go.Scatter(
            x=theta_grid,
            y=milestone_densities[step],
            mode="lines",
            name=f"k = {step}"
        )
    )

fig_density.add_vline(
    x=theta_true,
    line_dash="dash",
    annotation_text=(
        "True stiffness = 0.68"
    )
)

fig_density.update_layout(
    title=(
        "Sequential Posterior Density "
        "for Remaining Structural Stiffness"
    ),
    xaxis_title=(
        "Remaining stiffness efficiency, \u03b8"
    ),
    yaxis_title="Posterior density",
    template="plotly_white",
    width=1000,
    height=600
)

fig_density.show()


# =========================================================
# 11. Plot 2: estimator timeline
# =========================================================
fig_timeline = go.Figure()

fig_timeline.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig_timeline.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig_timeline.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text=(
        "True stiffness = 0.68"
    ),
    annotation_position="top right"
)

fig_timeline.update_layout(
    title=(
        "Sequential Estimation of "
        "Remaining Structural Stiffness"
    ),
    xaxis_title=(
        "Number of sensor readings, k"
    ),
    yaxis_title=(
        "Estimated stiffness efficiency"
    ),
    template="plotly_white",
    width=1000,
    height=550
)

fig_timeline.update_xaxes(
    tickmode="linear",
    dtick=1
)

fig_timeline.update_yaxes(
    range=[0, 1]
)

fig_timeline.show()


,Step,Noise epsilon,Sensor reading,Posterior mean,MAP,Posterior variance,Posterior SD,Mean error,MAP error
0,1,0.04571,35.59012,0.79468,0.79721,0.00858,0.09263,0.11468,0.11721
1,2,-0.15600,29.08908,0.69767,0.68769,0.00517,0.07189,0.01767,0.00769
2,3,0.11257,38.05103,0.71774,0.71066,0.00371,0.06090,0.03774,0.03066
3,4,0.14108,39.15175,0.73317,0.72770,0.00292,0.05407,0.05317,0.04770
4,5,-0.29266,25.37350,0.68229,0.67799,0.00206,0.04543,0.00229,0.00201
5,6,-0.19533,27.96723,0.66036,0.65680,0.00162,0.04024,0.01964,0.02320
6,7,0.01918,34.65828,0.66491,0.66175,0.00141,0.03754,0.01509,0.01825
7,8,-0.04744,32.42482,0.66286,0.66016,0.00123,0.03503,0.01714,0.01984
8,9,-0.00252,33.91442,0.66455,0.66214,0.00110,0.03312,0.01545,0.01786
9,10,-0.12796,29.91631,0.65766,0.65541,0.00097,0.03111,0.02234,0.02459


Initial prior mean: 0.8421
Initial prior MAP: 0.9333
Final posterior mean: 0.6873
Final MAP estimate: 0.6857
Final posterior variance: 0.000705
Final posterior SD: 0.0266
First confidence step: 5
Final 95% credible interval: (np.float64(0.6367), np.float64(0.7408))
P(theta < 0.75 | data): 0.9887


## 7. Reproducible Numerical Results

Using `RANDOM_SEED = 42`, the milestone results are approximately:

| $k$ | Posterior mean | MAP | Posterior SD |
|---:|---:|---:|---:|
| 0 | 0.8421 | 0.9333 | 0.1125 |
| 1 | 0.7947 | 0.7972 | 0.0926 |
| 2 | 0.6977 | 0.6877 | 0.0719 |
| 5 | 0.6823 | 0.6780 | 0.0454 |
| 10 | 0.6577 | 0.6554 | 0.0311 |
| 15 | 0.6873 | 0.6857 | 0.0266 |

Final posterior mean:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(15)} \approx 0.6873$$

Final MAP:

$$\widehat{\theta}_{\mathrm{MAP}}^{(15)} \approx 0.6857$$

Final posterior variance:

$$V_{15} \approx 0.000705$$

Final posterior standard deviation:

$$s_{15} \approx 0.0266$$

Approximate final 95% credible interval:

$$\Theta \mid \mathbf{y}^{(15)} \in [0.6367,\ 0.7408]$$

with posterior probability approximately 0.95.


## 8. Number of Readings Required

Confidently isolate requires a numerical rule. Use

$$\left|\widehat{\theta}_{\mathrm{Bayes}}^{(k)} - 0.68\right| \leq 0.02, \qquad \left|\widehat{\theta}_{\mathrm{MAP}}^{(k)} - 0.68\right| \leq 0.02, \qquad s_k \leq 0.05$$

For seed 42, the first step satisfying all conditions is

$$k = 5$$

At $k=5$,

$$\widehat{\theta}_{\mathrm{Bayes}}^{(5)} \approx 0.6823, \qquad \widehat{\theta}_{\mathrm{MAP}}^{(5)} \approx 0.6780, \qquad s_5 \approx 0.0454$$

Therefore, approximately five measurements overcome the optimistic healthy prior in this simulation.

The exact number varies with $\sigma$, $\alpha_0$, $\beta_0$, $\theta_{\mathrm{true}}$, and $\epsilon_1,\ldots,\epsilon_n$.


## 9. Safety-Threshold Interpretation

Let $\theta_{\mathrm{critical}}$ be the minimum acceptable stiffness. The posterior probability of an unsafe state is

$$P(\Theta < \theta_{\mathrm{critical}} \mid \mathbf{y}^{(k)}) = \int_0^{\theta_{\mathrm{critical}}} f_k(\theta)\, d\theta$$

For example, $\theta_{\mathrm{critical}} = 0.75$. The final simulation gives approximately

$$P(\Theta < 0.75 \mid \mathbf{y}^{(15)}) \approx 0.9887$$

Thus, there is approximately 98.87% posterior probability that stiffness is below 75%.

As $k$ increases, $\operatorname{Var}(\Theta \mid \mathbf{y}^{(k)}) \downarrow$, so posterior density narrows $\Rightarrow$ uncertainty decreases $\Rightarrow$ safety decisions become more decisive.

A narrow posterior completely below a safety threshold implies

$$P(\Theta < \theta_{\mathrm{critical}} \mid \mathbf{y}) \approx 1$$

supporting inspection, repair, load restriction, or replacement.
